In [ ]:
%py
# This script masks the last 4 digits of the invoice_number column in the d_product_revenue_clone table

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat, lit, substring
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, DateType

# Initialize Spark session
spark = SparkSession.builder.appName("MaskInvoiceNumber").getOrCreate()

# Define schema for d_product_revenue
schema = StructType([
    StructField("product_id", LongType(), True),
    StructField("product_name", StringType(), True),
    StructField("product_type", StringType(), True),
    StructField("revenue", LongType(), True),
    StructField("country", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("purchased_date", DateType(), True),
    StructField("invoice_date", DateType(), True),
    StructField("invoice_number", LongType(), True),
    StructField("is_returned", LongType(), True),
    StructField("customer_satisfaction_score", LongType(), True),
    StructField("product_details", StringType(), True),
    StructField("customer_first_purchased_date", DateType(), True),
    StructField("customer_first_product", StringType(), True),
    StructField("customer_first_revenue", DoubleType(), True)
])

try:
    # Drop the d_product_revenue_clone table if exists
    spark.sql("DROP TABLE IF EXISTS purgo_playground.d_product_revenue_clone")

    # Create a replica of d_product_revenue table
    spark.sql("""
        CREATE TABLE purgo_playground.d_product_revenue_clone AS
        SELECT * FROM purgo_playground.d_product_revenue
    """)

    # Load the replica table into a DataFrame
    df_clone = spark.table("purgo_playground.d_product_revenue_clone")

    # Mask the last 4 digits of invoice_number by converting it to STRING
    masked_df = df_clone.withColumn(
        "invoice_number",
        concat(substring(col("invoice_number").cast(StringType()), 1, 6), lit("****"))
    )

    # Write the DataFrame back to the clone table
    masked_df.write.mode("overwrite").saveAsTable("purgo_playground.d_product_revenue_clone")

except Exception as e:
    print(f"Error during masking process: {e}")

# Stop the Spark session
spark.stop()